# 発展のノートC：SAM 3D Body を無料の Colab で動かす

本編の MediaPipe より大きなモデル、SAM 3D Body（Meta、2025年）で肩挙上角を測り、正解（モーションキャプチャー）と MediaPipe と比べます。

## 動かす前に

- ランタイムを GPU にします。メニューの［ランタイム］→［ランタイムのタイプを変更］で「T4 GPU」を選びます（無料の Colab で選べます）。
- Hugging Face（AIのモデルを配るサイト）で、facebook/sam-3d-body-dinov3 の利用申請が承認されている必要があります。手順は「事前準備の手順書」（docs/handout-hf-setup.md）にあります。
- 承認がまだ（審査待ち）のアカウントでは、4のセルのあとで「403」「awaiting a review」と出て止まります。承認のメールを待ってから押し直してください。
- 重み（約2.8GB）の取得に数分、正面の動画1本の処理に約5分かかります。

## このノートの使い方（本編と同じ）

- 上のセルから順に、全部押してください。下のセルは、上のセルで作ったものを使います。
- 中身を読み飛ばしてよいセルには、そう書いてあります。

## 使うデータ

REHAB24-6（Masaryk大学、CC BY-NC 4.0）の9番の人の、右腕の外転。正面・斜め・横から撮った動画と、モーションキャプチャーの正解です。患者のデータではありません。

## 1. GPU を確かめる

GPU がないと、このノートは動きません。「GPU がありません」と出たら、上の「動かす前に」のとおりランタイムを変えて、このセルから押し直してください。

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit('GPU がありません。［ランタイム］→［ランタイムのタイプを変更］で T4 GPU を選んでください')
print('GPU:', gpu)

## 2. 道具を入れる

1行目は、MediaPipe と、SAM 3D Body が使う道具を入れる行です。2行目は SAM 3D Body のプログラムを持ってくる行、3行目は MediaPipe のモデルを持ってくる行です。2〜3分かかります。

In [ ]:
!pip install -q mediapipe==0.10.35 matplotlib-fontja pytorch-lightning roma yacs einops braceexpand omegaconf trimesh
!git clone -q https://github.com/facebookresearch/sam-3d-body.git && git -C sam-3d-body checkout -q b5c765a0d89d789985e186d396315e7590887b94
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
import sys
sys.path.insert(0, '/content/sam-3d-body')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
print('道具の準備ができました')

## 3. Hugging Face のトークンを貼る

セルを押すと入力欄が出ます。承認されたアカウントのアクセストークン（`hf_`で始まる文字列）を貼って、Enter を押します。貼った文字は表示されません。
最後に、入ったアカウントの名前が出ます。申請したアカウントと同じか確かめてください。

In [ ]:
import getpass, os
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face のトークン: ')
from huggingface_hub import whoami
print('入ったアカウント:', whoami()['name'])

## 4. 動画と正解を持ってくる

本編と同じ動画です。

In [ ]:
URL = 'https://k0heiun0.github.io/pt_seminar/data/rehab24/'
!wget -q {URL}front.mp4 {URL}oblique.mp4 {URL}side.mp4
!wget -q {URL}front_truth.csv {URL}oblique_truth.csv {URL}side_truth.csv
print('データを持ってきました')

## 5. モデルと関数を作る（中身は読み飛ばしてよい）

最初に SAM 3D Body のモデルを読み込みます（重みの取得に数分）。そのあと関数を作ります。

| 関数 | 役割 | 入力 | 出力 |
|---|---|---|---|
| `shoulder_elevation` | 肩挙上角を出す（本編と同じ定義） | 1コマの点、使う点の番号 | 角度（度） |
| `track_both` | MediaPipe と SAM 3D Body で点を出す | 動画の名前、何コマに1回か | コマの番号、MediaPipe の2Dと3Dの点、SAM の3Dの点 |
| `compare` | 3つの肩挙上角を出し、正解と並べる | 動画の名前 | 正解と3つの角度 |
| `summary` | 正解との差を3つの見方で表示する | `compare`の結果 | （表示が出る） |

MediaPipe と SAM 3D Body では、点の番号が違います。`MP`と`SAM`は、左肩・右肩・右肘・左股・右股の番号です。

In [ ]:
import contextlib, io
import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision
from sam_3d_body import load_sam_3d_body_hf, SAM3DBodyEstimator

model, cfg = load_sam_3d_body_hf('facebook/sam-3d-body-dinov3', device='cuda')
sam_model = SAM3DBodyEstimator(sam_3d_body_model=model, model_cfg=cfg)

MP = (11, 12, 14, 23, 24)
SAM = (5, 6, 8, 9, 10)


def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    cos_angle = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def shoulder_elevation(p, points):
    left_shoulder, shoulder, elbow, left_hip, hip = points
    upper_arm = p[elbow] - p[shoulder]
    trunk = (p[left_hip] + p[hip]) / 2 - (p[left_shoulder] + p[shoulder]) / 2
    return angle_between_vectors(upper_arm, trunk)


def track_both(file, step=3):
    # 役割：stepコマに1回、MediaPipe の点から人の枠を作り、SAM 3D Body に渡す
    options = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
        running_mode=vision.RunningMode.VIDEO, num_poses=1)
    video = cv2.VideoCapture(file)
    fps = video.get(cv2.CAP_PROP_FPS)
    frames, mp2d, mp3d, sam3d = [], [], [], []
    k = 0
    with vision.PoseLandmarker.create_from_options(options) as pose:
        while True:
            ok, frame = video.read()
            if not ok:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = rgb.shape[:2]
            result = pose.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
                                           int(k * 1000 / fps))
            if k % step == 0 and result.pose_landmarks:
                xy = np.array([[p.x * w, p.y * h] for p in result.pose_landmarks[0]])
                (x1, y1), (x2, y2) = xy.min(axis=0), xy.max(axis=0)
                cx, cy, s = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * 0.65
                box = np.array([[max(cx - s, 0), max(cy - s, 0), min(cx + s, w), min(cy + s, h)]])
                with contextlib.redirect_stdout(io.StringIO()):
                    out = sam_model.process_one_image(rgb, bboxes=box, inference_type='body')
                frames.append(k)
                mp2d.append(xy)
                mp3d.append([[p.x, p.y, p.z] for p in result.pose_world_landmarks[0]])
                sam3d.append(out[0]['pred_keypoints_3d'])
                if len(frames) % 100 == 0:
                    print(len(frames), 'コマ処理しました')
            k += 1
    return np.array(frames), np.array(mp2d), np.array(mp3d), np.array(sam3d)


def compare(name, step=3):
    frames, mp2d, mp3d, sam3d = track_both(name + '.mp4', step)
    truth = np.loadtxt(name + '_truth.csv', delimiter=',', skiprows=1, usecols=2)
    keep = frames < len(truth)  # 正解のないコマは比べない
    return {'frames': frames[keep], 'truth': truth[frames[keep]],
            'MediaPipe 2D': np.array([shoulder_elevation(p, MP) for p in mp2d[keep]]),
            'MediaPipe 3D': np.array([shoulder_elevation(p, MP) for p in mp3d[keep]]),
            'SAM 3D Body': np.array([shoulder_elevation(p, SAM) for p in sam3d[keep]])}


def summary(result):
    print('コマ数', len(result['frames']))
    for name in ['MediaPipe 2D', 'MediaPipe 3D', 'SAM 3D Body']:
        d = result[name] - result['truth']
        print(name, '差', round(np.mean(np.abs(d)), 1), '一定のずれ', round(np.mean(d), 1),
              'ずれを引いた残り', round(np.mean(np.abs(d - np.mean(d))), 1))

## 6. 正面の動画で比べる（約5分）

3コマに1回、MediaPipe と SAM 3D Body で点を出し、肩挙上角を正解と比べます。数は3つの見方で出します。

- 差：コマごとの差の大きさの平均（本編と同じ）
- 一定のずれ：差の向きも入れた平均。いつも同じだけ大きく（小さく）出る分
- ずれを引いた残り：一定のずれを引いたあとの差。角度の動きの形がどれだけ合っているか

In [ ]:
front = compare('front')
summary(front)

## 7. 時系列のグラフ

正解・MediaPipe 3D・SAM 3D Body を重ねます。

In [ ]:
t = front['frames'] / 30
plt.plot(t, front['truth'], color='black', label='正解')
plt.plot(t, front['MediaPipe 3D'], label='MediaPipe 3D')
plt.plot(t, front['SAM 3D Body'], label='SAM 3D Body')
plt.xlabel('時間 [秒]')
plt.ylabel('肩挙上角 [度]')
plt.legend()
plt.show()

## 8. 一定のずれを引いて重ねる

SAM 3D Body の角度から一定のずれを引くと、正解とどれくらい重なるかを見ます。

In [ ]:
offset = np.mean(front['SAM 3D Body'] - front['truth'])
plt.plot(t, front['truth'], color='black', label='正解')
plt.plot(t, front['SAM 3D Body'] - offset, label='SAM 3D Body（ずれを引いた）')
plt.xlabel('時間 [秒]')
plt.ylabel('肩挙上角 [度]')
plt.legend()
plt.show()
print('引いたずれ', round(offset, 1))

## 9. 撮る向きを変える（任意、1本4〜5分）

斜めと横から撮った動画でも、同じように比べます。時間がなければ飛ばしてかまいません。

In [ ]:
for name in ['oblique', 'side']:
    print('=====', name)
    summary(compare(name))

## 10. 問い

- 正面で、SAM 3D Body の「差」が MediaPipe より大きいのはなぜか。「一定のずれ」と「ずれを引いた残り」から考える。
- 同じ人を同じ撮り方で前後に撮って比べるとき、一定のずれはどう扱えばよいか。
- 横から撮ると、大きなモデルでも外転を測れないのはなぜか。